In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Superstore Data Processing")
    .getOrCreate()
)

print("Spark Session Created Successfully!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 08:02:35 WARN Utils: Your hostname, Ubantu, resolves to a loopback address: 127.0.1.1; using 192.168.31.159 instead (on interface wlo1)
26/06/29 08:02:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 08:02:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully!


In [3]:
df = spark.read.csv(
    "../data/raw/Superstore.csv",
    header=True,
    inferSchema=True
)

In [4]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [5]:
print("Rows :", df.count())

Rows : 9994


In [6]:
print("Columns :", len(df.columns))

Columns : 21


In [7]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [8]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [9]:
df.describe().show()

26/06/29 08:19:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 6:>                                                          (0 + 1) / 1]

+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|summary|            Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|    City|  State|       Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|  count|              9994|          9994|   

In [10]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [12]:
df.select("Sales").distinct().show(20, truncate=False)

+-------+
|Sales  |
+-------+
|1.248  |
|7.16   |
|258.072|
|961.48 |
|647.84 |
|78.272 |
|105.42 |
|119.8  |
|406.6  |
|197.97 |
|79.14  |
|251.79 |
|219.84 |
|710.832|
|526.45 |
|108.576|
|123.136|
|203.92 |
|35.568 |
|767.214|
+-------+
only showing top 20 rows


In [13]:
from pyspark.sql.functions import col

clean_df = (
    df.withColumn("Sales", col("Sales").cast("double"))
      .withColumn("Quantity", col("Quantity").cast("int"))
      .withColumn("Discount", col("Discount").cast("double"))
)

In [14]:
clean_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [15]:
clean_df.select(
    "Sales",
    "Quantity",
    "Discount"
).show(5)

+--------+--------+--------+
|   Sales|Quantity|Discount|
+--------+--------+--------+
|  261.96|       2|     0.0|
|  731.94|       3|     0.0|
|   14.62|       2|     0.0|
|957.5775|       5|    0.45|
|  22.368|       2|     0.2|
+--------+--------+--------+
only showing top 5 rows


In [16]:
selected_df = clean_df.select(
    "Order ID",
    "Order Date",
    "Customer Name",
    "Category",
    "Sales",
    "Profit",
    "Region"
)

selected_df.show(5, truncate=False)

+--------------+----------+---------------+---------------+--------+--------+------+
|Order ID      |Order Date|Customer Name  |Category       |Sales   |Profit  |Region|
+--------------+----------+---------------+---------------+--------+--------+------+
|CA-2016-152156|11/8/2016 |Claire Gute    |Furniture      |261.96  |41.9136 |South |
|CA-2016-152156|11/8/2016 |Claire Gute    |Furniture      |731.94  |219.582 |South |
|CA-2016-138688|6/12/2016 |Darrin Van Huff|Office Supplies|14.62   |6.8714  |West  |
|US-2015-108966|10/11/2015|Sean O'Donnell |Furniture      |957.5775|-383.031|South |
|US-2015-108966|10/11/2015|Sean O'Donnell |Office Supplies|22.368  |2.5164  |South |
+--------------+----------+---------------+---------------+--------+--------+------+
only showing top 5 rows


In [17]:
high_sales_df = clean_df.filter(col("Sales") > 500)

high_sales_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+---------------+----------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name| Segment|      Country|           City|     State|Postal Code|Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+---------------+----------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+--------+
|     2|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute|Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-CH-10000454| Furniture|      Chairs|Hon Deluxe Fabric...|  731.94|       3

In [18]:
west_region_df = clean_df.filter(
    col("Region") == "West"
)

west_region_df.show(5)

+------+--------------+----------+---------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+---------+--------------+-----------+---------------+---------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|     3|CA-2016-138688| 6/12/2016|6/16/2016|  Second Class|   DV-13045|Darrin Van Huff|Corporate|United States|Los Angeles|California|      90036|  West|OFF-LA-10000240|Office Supplies|      Labels|Self-Adhesive Add...|  14.62|       2|  

In [19]:
filtered_df = clean_df.filter(
    (col("Sales") > 500) &
    (col("Profit") > 100)
)

filtered_df.show(5)

+------+--------------+----------+----------+--------------+-----------+--------------+---------+-------------+-------------+--------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID| Customer Name|  Segment|      Country|         City|   State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+--------------+---------+-------------+-------------+--------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     2|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|   Claire Gute| Consumer|United States|    Henderson|Kentucky|      42420|  South|FUR-CH-10000454|      Furniture|      Chairs|Hon Deluxe Fabric...|  731.94|

In [20]:
renamed_df = clean_df.withColumnRenamed(
    "Sales",
    "Total_Sales"
)

renamed_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|Total_Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+-----------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush 

In [22]:
from pyspark.sql.functions import round

transformed_df = renamed_df.withColumn(
    "Profit_Margin",
    round((col("Profit") / col("Total_Sales")) * 100, 2)
)

transformed_df.select(
    "Total_Sales",
    "Profit",
    "Profit_Margin"
).show(10)

+-----------+--------+-------------+
|Total_Sales|  Profit|Profit_Margin|
+-----------+--------+-------------+
|     261.96| 41.9136|         16.0|
|     731.94| 219.582|         30.0|
|      14.62|  6.8714|         47.0|
|   957.5775|-383.031|        -40.0|
|     22.368|  2.5164|        11.25|
|      48.86| 14.1694|         29.0|
|       7.28|  1.9656|         27.0|
|    907.152| 90.7152|         10.0|
|     18.504|  5.7825|        31.25|
|      114.9|   34.47|         30.0|
+-----------+--------+-------------+
only showing top 10 rows


In [24]:
clean_df.select(
    "Product Name",
    "Sales"
).show(20, truncate=False)

+----------------------------------------------------------------------------+--------+
|Product Name                                                                |Sales   |
+----------------------------------------------------------------------------+--------+
|Bush Somerset Collection Bookcase                                           |261.96  |
|Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back                 |731.94  |
|Self-Adhesive Address Labels for Typewriters by Universal                   |14.62   |
|Bretford CR4500 Series Slim Rectangular Table                               |957.5775|
|Eldon Fold 'N Roll Cart System                                              |22.368  |
|Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood            |48.86   |
|Newell 322                                                                  |7.28    |
|Mitel 5320 IP Phone VoIP phone                                              |907.152 |
|DXL Angle-View Binders with Loc

In [26]:
df.show(5, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                               |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute    |Consumer |Un

In [30]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("../data/raw/Superstore.csv")